In [ ]:
import odc.stac
from pystac import Catalog
from shapely.geometry import shape, box
from shapely import intersects
from shapely import wkt
from pyproj import Transformer
from shapely.ops import transform

import json
from pathlib import Path

import deltalake
import lonboard
import pyarrow as pa
import pyarrow.parquet as pq
import pystac_client

import stac_geoparquet

In [17]:
aoi_wkt = "POLYGON ((1514969.7099272856 5209247.52503516, 1514391.8229496903 5209385.117172683, 1513621.3069795633 5209013.618401371, 1513428.6779870314 5207623.937812392, 1515176.09813357 5204541.873931885, 1516662.0932188153 5201679.957471414, 1518478.3094341147 5199395.927988537, 1522330.8892847502 5197854.896048282, 1523865.041618128 5197476.517670095, 1524697.4740501402 5197731.063124511, 1525041.454393947 5198075.043468318, 1525127.4494799003 5199010.6700034775, 1524711.2332638926 5199836.222828608, 1523183.9605373908 5200207.72159992, 1520349.5625044233 5202725.657716585, 1518712.2160679032 5204913.372703196, 1517707.7934639875 5206220.498009661, 1516111.7246687242 5207899.122087438, 1514969.7099272856 5209247.52503516))"
aoi = shape(wkt.loads(aoi_wkt))

In [18]:
transformer = Transformer.from_crs("EPSG:2193", "EPSG:4326", always_xy=True)

In [19]:
nz_catalog = Catalog.from_file(
    "https://nz-elevation.s3-ap-southeast-2.amazonaws.com/catalog.json"
)

In [20]:
nz_catalog

<Catalog id=nz-elevation>

In [21]:
collection = next(nz_catalog.get_all_collections())

collection


<Collection id=01HQS6C8F88JGM04345RY84N9D>

In [22]:
item = next(collection.get_all_items())

item

<Item id=AY30_10000_0405>

In [ ]:
ds = odc.stac.load(
    [item],
    crs="EPSG:2193",
    resolution=100,
    geopolygon=transform(transformer.transform, aoi),
)
ds

<xarray.Dataset> Size: 15kB
Dimensions:      (y: 72, x: 48, time: 1)
Coordinates:
  * y            (y) float64 576B 5.996e+06 5.996e+06 ... 5.989e+06 5.989e+06
  * x            (x) float64 384B 1.727e+06 1.727e+06 ... 1.732e+06 1.732e+06
  * time         (time) datetime64[ns] 8B 2016-08-15T12:00:00
    spatial_ref  int32 4B 2193
Data variables:
    visual       (time, y, x) float32 14kB nan nan nan nan ... 18.2 14.6 6.6 1.2

In [ ]:
nz_elevation_catalog = Catalog.from_file(
    "https://nz-elevation.s3-ap-southeast-2.amazonaws.com/catalog.json"
)

# bbox = box(171.9297, -43.3651, 172.0715, -43.2836)


def search_items(
    catalog,
    regions=None,
    collections=None,
    geospatial_category=None,
    geom=None,
    max_items=10,
    sortby=None,
):
    candidate_collections = catalog.get_all_collections()

    # filter out collections based on criteria
    if regions:
        candidate_collections = [
            c
            for c in candidate_collections
            if c.extra_fields.get("linz:region") in regions
        ]
    if collections:
        candidate_collections = [
            c
            for c in candidate_collections
            if c.extra_fields.get("linz:slug") in collections
        ]
    if geospatial_category:
        candidate_collections = [
            c
            for c in candidate_collections
            if c.extra_fields.get("linz:geospatial_category") == geospatial_category
        ]

    print(f"Found {len(candidate_collections)} candidate collections after filtering.")

    items = []
    for collection in candidate_collections:
        for item in collection.get_all_items():
            if geom and not intersects(shape(item.geometry), geom):
                continue
            items.append(item)
            if len(items) >= max_items:
                break
        if len(items) >= max_items:
            break

    return items


items = search_items(
    nz_elevation_catalog,
    regions=["auckland"],
    collections=["auckland-north_2016-2018"],
    geospatial_category="dem",
    # geom=bbox,
    max_items=1,
    sortby="-datetime",
)


items

Found 1 candidate collections after filtering.


[<Item id=AY30_10000_0405>]

In [11]:
ds = odc.stac.load(
    items,
    crs="EPSG:2193",
    resolution=100,
)
ds

<xarray.Dataset> Size: 15kB
Dimensions:      (y: 72, x: 48, time: 1)
Coordinates:
  * y            (y) float64 576B 5.996e+06 5.996e+06 ... 5.989e+06 5.989e+06
  * x            (x) float64 384B 1.727e+06 1.727e+06 ... 1.732e+06 1.732e+06
  * time         (time) datetime64[ns] 8B 2016-08-15T12:00:00
    spatial_ref  int32 4B 2193
Data variables:
    visual       (time, y, x) float32 14kB nan nan nan nan ... 18.2 14.6 6.6 1.2

In [12]:
from pystac_client import Client

nz_elevation_client = Client.open(
    "https://nz-elevation.s3-ap-southeast-2.amazonaws.com/catalog.json"
)

/home/quinnhornblow/repositories/linz-s3-utils/.venv/lib/python3.11/site-packages/pystac_client/client.py:191: NoConformsTo: Server does not advertise any conformance classes.
  warnings.warn(NoConformsTo())


In [ ]:
nz_elevation_client.search(
    collections=["auckland-north_2016-2018"],
)

DoesNotConformTo: Server does not conform to ITEM_SEARCH, There is no fallback option available for search.